# Student smoke-test notebook

Walks through every student feature, including the GDPR data-rights commands.
Pairs with `teacher-notebook-example.ipynb` — run that first to create the
lesson and grab the join code.

**Before you start:**
- Replace `<JOIN_CODE>` below with the code printed by the teacher's
  `%cadence_create_lesson` cell.
- Watch the dashboard (also printed by the teacher) in another tab to see
  attempts and submissions arrive in real time.

In [ ]:
%load_ext cadence

## 1. Join the session

`%cadence_session` shows the **GDPR Article 13 just-in-time notice** before
sending anything to the server. Read what it says: what's collected, who
sees it, retention, your rights, and the privacy contact.

You can use a pseudonym (like `birb_42`) instead of a real name — the
notice reminds you of that. The display name is shown to your teacher.

In [ ]:
%cadence_session <JOIN_CODE> "smoke_student"

## 2. Solve some checkpoints

The teacher registered four auto-checked checkpoints plus a manual one.
Each `check()` call sends only the **value** to the server — your code
stays on your machine unless you use `%%cadence_submit`.

In [ ]:
from cadence import check

check("hello", "Hello, World!")    # exact comparator

In [ ]:
import numpy as np

arr = np.arange(100)
check("mean", float(arr.mean()))    # numeric, tolerance 0.001

In [ ]:
grid = np.arange(12).reshape(3, 4)
check("row-sums", list(grid.sum(axis=1)))   # set comparator: order-insensitive

## 3. Hint flow

Submit a wrong answer, then ask for the hint. Hints are gated by
`--hint-after-attempts` — default 1, so it's available immediately after
the first wrong try.

In [ ]:
check("higgs-peak", 130)    # intentionally wrong

In [ ]:
import cadence
cadence.show_hint("higgs-peak")

## 4. Solution reveal

After `--reveal-after` attempts (3 in this case), the student can request
the worked solution. Two more wrong attempts first, then reveal.

In [ ]:
check("higgs-peak", 128)    # wrong #2

In [ ]:
check("higgs-peak", 124)    # wrong #3 — now eligible for the solution

In [ ]:
cadence.show_solution("higgs-peak")

In [ ]:
check("higgs-peak", 125)    # the right answer

## Reflection checkpoint (manual)

Some checkpoints are reflection prompts — no single right answer. The teacher
registered them with `--comparator manual` and a hint that's actually the
prompt. You self-attest by running `cadence.mark_done("<id>")`.

The teacher sees you've completed it on the dashboard but does not see any
specific answer — that part is between you and your notebook.

In [ ]:
# Write whatever reflection your teacher asked for in a regular markdown
# cell or as a comment here — Cadence won't read it. Then mark the
# checkpoint as done so it shows up green on the dashboard.
cadence.mark_done("reflect")

## 5. Timed check

`%%cadence_time <id>` runs the cell, times it, then submits the last
expression's value. The elapsed time appears on the teacher's dashboard
as a timing sample.

In [ ]:
%%cadence_time mean
import numpy as np
arr = np.arange(100)
float(arr.mean())

## 6. Code submission

Only works on checkpoints with `--allow-submissions`. Cell runs normally
AND the source is shipped to the teacher's dashboard for review.

In [ ]:
%%cadence_submit higgs-peak
import numpy as np
bin_edges = np.arange(100, 151)
# imagine some m_gg data here
counts = np.random.poisson(50, len(bin_edges) - 1)
peak = int(bin_edges[np.argmax(counts)])
peak

## 7. Your data rights — view, export, delete

Three Jupyter magic commands for the GDPR Article 15 / 17 / 20 trio.
All three work against the active session — no other credential needed.

### 7a. See everything Cadence has stored about you (Article 15)

Returns a styled table of attempts, code submissions, and solution reveals
for this session.

In [ ]:
%cadence_my_data

### 7b. Download your data as JSON (Article 20)

Writes a file to the current directory by default; pass `--path` to choose
where. Open it in any editor — it's machine-readable JSON suitable for
feeding into other tools.

In [ ]:
%cadence_export_my_data

### 7c. Delete everything Cadence has stored about you (Article 17)

First run shows the confirmation card. Re-run with `--yes` to actually
wipe. Once deleted, the session is gone from the server and the kernel's
session state is cleared — to keep using Cadence you'd need to
`%cadence_session` again with a new display name.

In [ ]:
%cadence_delete_my_data    # shows the confirmation block; doesn't delete

In [ ]:
%cadence_delete_my_data --yes    # actually wipes the session

## Verify

1. **In this kernel:** any further `check()` call should fail with "No active session".
2. **On the dashboard:** the `smoke_student` row should disappear from the roster.
3. **Access log:** the backend should have a `delete_session` entry. The teacher
   smoke test verifies that with a direct DB query — see `docs/smoke-test.md` §8.

If all three are true, the student-side smoke test passes.